# Self-Supervised Pretraining (SimCLR) + Fine-Tuning

**Phase 1 — SSL Pretraining** (unsupervised, utilise train + test = 3150 images)
- Encodeur SE-ResNet34 + tête de projection MLP
- Loss NT-Xent (contrastive, temperature=0.07)
- Augmentations agressives : random crops, flips, blur, contrast
- Aucun label utilisé

**Phase 2 — Fine-Tuning** (supervisé, 5-fold CV sur les 2361 images étiquetées)
- Charge les poids pré-entraînés
- 10 epochs linear probe (backbone gelé) → 140 epochs full fine-tune
- TTA + soumission Kaggle

## 1. Setup

In [1]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = ROOT / 'data'
OUTPUT_DIR     = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'
FIGURE_DIR     = OUTPUT_DIR / 'figures'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
GPU   : NVIDIA RTXA6000-48Q
VRAM  : 51.3 GB


## 2. Données

Phase SSL : **toutes les images** (train + test, sans labels).
Phase fine-tune : seulement les **images étiquetées** du train.

In [2]:
train_csv = DATA_DIR / 'train.csv'
train_dir = DATA_DIR / 'train'
test_dir  = DATA_DIR / 'test'

df = pd.read_csv(train_csv, sep=';')
df['id']    = df['id'].astype(int)
df['label'] = df['label'].astype(int)
df['path']  = df['id'].map(lambda x: train_dir / f'{x}.tif')

train_paths = list(train_dir.glob('*.tif'))
test_paths  = list(test_dir.glob('*.tif'))
all_paths   = train_paths + test_paths

print(f'Train (labeled) : {len(train_paths)}')
print(f'Test (unlabeled): {len(test_paths)}')
print(f'Total SSL pool  : {len(all_paths)}')

Train (labeled) : 2361
Test (unlabeled): 789
Total SSL pool  : 3150


## 3. Augmentations SSL + Dataset SimCLR

Chaque image donne **deux vues augmentées** `(v1, v2)`. Le modèle doit apprendre que v1 et v2 viennent de la même image, contre toutes les autres images du batch.

In [3]:
SSL_CROP = 224   # carré — les textures sont isotropes

ssl_tfms = transforms.Compose([
    transforms.RandomResizedCrop(SSL_CROP, scale=(0.2, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.4, contrast=0.4)
    ], p=0.8),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))
    ], p=0.5),
    transforms.RandomRotation(degrees=45),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


class SimCLRDataset(Dataset):
    """Retourne (view1, view2) — deux augmentations de la même image."""
    def __init__(self, paths, transform):
        self.paths     = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('L')
        return self.transform(img), self.transform(img)

## 4. Modèle SimCLR

- **`SEResNetEncoder`** : backbone SE-ResNet34 → vecteur (B, 512)
- **`ProjectionHead`** : MLP 512 → 256 → 128 (avec BatchNorm) → L2-normalisé
- **`SimCLRModel`** : encoder + projector (utilisé seulement en pré-entraînement)

In [4]:
# ── SE Block ──────────────────────────────────────────────────────────

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid(),
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)


# ── SE-ResNet basic block ─────────────────────────────────────────────

class SEResNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.se    = SEBlock(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.down  = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
    def forward(self, x):
        skip = x if self.down is None else self.down(x)
        out  = self.relu(self.bn1(self.conv1(x)))
        return self.relu(self.se(self.bn2(self.conv2(out))) + skip)


# ── SE-ResNet34 Encoder (sans tête de classification) ─────────────────

class SEResNetEncoder(nn.Module):
    """Retourne (B, 512) — pas de dropout, pas de classif."""
    def __init__(self, in_channels=1):
        super().__init__()
        self._ch = 64
        self.stem   = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make(64,  3, stride=1)
        self.layer2 = self._make(128, 4, stride=2)
        self.layer3 = self._make(256, 6, stride=2)
        self.layer4 = self._make(512, 3, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 512

    def _make(self, out_ch, n, stride):
        layers = [SEResNetBlock(self._ch, out_ch, stride=stride)]
        self._ch = out_ch
        for _ in range(1, n):
            layers.append(SEResNetBlock(self._ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.pool(x).flatten(1)


# ── Projection Head ───────────────────────────────────────────────────

class ProjectionHead(nn.Module):
    """MLP 512→256→128 avec BN. Sortie L2-normalisée."""
    def __init__(self, in_dim=512, hidden=256, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, h):
        return F.normalize(self.net(h), dim=1)


# ── Modèle complet SimCLR ─────────────────────────────────────────────

class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder   = SEResNetEncoder(in_channels=1)
        self.projector = ProjectionHead(512, 256, 128)
    def forward(self, x):
        return self.projector(self.encoder(x))


# ── NT-Xent Loss ──────────────────────────────────────────────────────

class NTXentLoss(nn.Module):
    """
    NT-Xent (Normalized Temperature-scaled Cross Entropy) — SimCLR.
    z1, z2 : (B, D) embeddings L2-normalisés.
    Paire positive : (z1[i], z2[i]). Toutes les autres paires sont négatives.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, z1, z2):
        B   = z1.size(0)
        z   = torch.cat([z1, z2], dim=0)            # (2B, D)
        sim = torch.mm(z, z.T) / self.tau            # (2B, 2B)
        sim.fill_diagonal_(float('-inf'))             # exclut self-similarity
        # labels : pour i<B la positive est i+B, pour i>=B c'est i-B
        labels = torch.cat([
            torch.arange(B, 2*B),
            torch.arange(B)
        ]).to(z.device)
        return F.cross_entropy(sim, labels)


# Vérification rapide
m = SimCLRModel().to(DEVICE)
enc_params  = sum(p.numel() for p in m.encoder.parameters())
proj_params = sum(p.numel() for p in m.projector.parameters())
print(f'Encoder     : {enc_params:>12,} params')
print(f'Projector   : {proj_params:>12,} params')
del m
torch.cuda.empty_cache()

Encoder     :   21,441,144 params
Projector   :      164,736 params


## 5. Phase 1 — Pré-entraînement SSL

300 epochs sur les 3150 images (train + test, sans labels).
L'encodeur apprend à produire des représentations cohérentes pour les textures.

In [ ]:
SSL_EPOCHS     = 300
SSL_BATCH      = 128
SSL_LR         = 1e-3
SSL_TEMP       = 0.07
SSL_CKPT       = CHECKPOINT_DIR / 'ssl_encoder.pt'
SSL_NUM_WORKERS = 0

ssl_ds     = SimCLRDataset(all_paths, ssl_tfms)
ssl_loader = DataLoader(ssl_ds, batch_size=SSL_BATCH, shuffle=True,
                        num_workers=SSL_NUM_WORKERS, pin_memory=True, drop_last=True)

ssl_model  = SimCLRModel().to(DEVICE)
ssl_loss_fn = NTXentLoss(temperature=SSL_TEMP).to(DEVICE)
ssl_optim  = torch.optim.Adam(ssl_model.parameters(), lr=SSL_LR, weight_decay=1e-4)
ssl_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(ssl_optim, T_max=SSL_EPOCHS, eta_min=1e-5)

print(f'SSL dataset    : {len(ssl_ds)} images')
print(f'Steps per epoch: {len(ssl_loader)}')
print(f'Total steps    : {len(ssl_loader) * SSL_EPOCHS:,}')

ssl_history = []
best_ssl_loss = float('inf')
start = time.time()

for epoch in range(1, SSL_EPOCHS + 1):
    ssl_model.train()
    epoch_loss = 0.0
    for v1, v2 in tqdm(ssl_loader, leave=False):
        v1 = v1.to(DEVICE, non_blocking=True)
        v2 = v2.to(DEVICE, non_blocking=True)
        z1 = ssl_model(v1)
        z2 = ssl_model(v2)
        loss = ssl_loss_fn(z1, z2)
        ssl_optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ssl_model.parameters(), max_norm=5.0)
        ssl_optim.step()
        epoch_loss += loss.item()
    ssl_sched.step()

    avg_loss = epoch_loss / len(ssl_loader)
    ssl_history.append({'epoch': epoch, 'loss': avg_loss, 'lr': ssl_sched.get_last_lr()[0]})

    if avg_loss < best_ssl_loss:
        best_ssl_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'encoder_state_dict': ssl_model.encoder.state_dict(),
            'projector_state_dict': ssl_model.projector.state_dict(),
            'loss': avg_loss,
        }, SSL_CKPT)

    if epoch % 10 == 0 or epoch == 1:
        elapsed = (time.time() - start) / 60
        print(f'[SSL] ep {epoch:03d}/{SSL_EPOCHS} | loss {avg_loss:.4f} (best {best_ssl_loss:.4f}) | {elapsed:.1f} min')

elapsed = (time.time() - start) / 60
print(f'\nSSL pretraining done — {elapsed:.1f} min | best loss {best_ssl_loss:.4f}')
print(f'Checkpoint: {SSL_CKPT}')

del ssl_model
torch.cuda.empty_cache()

SSL dataset    : 3150 images
Steps per epoch: 24
Total steps    : 7,200


  0%|          | 0/24 [00:00<?, ?it/s]

[SSL] ep 001/300 | loss 4.8520 (best 4.8520) | 0.5 min


  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

...

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

[SSL] ep 050/300 | loss 2.9026 (best 2.8940) | 25.2 min


  0%|          | 0/24 [00:00<?, ?it/s]

In [5]:
ssl_df = pd.DataFrame(ssl_history)
ssl_df.to_csv(OUTPUT_DIR / 'ssl_history.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ssl_df['epoch'], ssl_df['loss'])
ax.set_xlabel('Epoch'); ax.set_ylabel('NT-Xent Loss')
ax.set_title('SSL Pretraining Loss')
ax.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ssl_loss.png', dpi=150)
plt.show()
print('Loss finale:', ssl_df['loss'].iloc[-1])

NameError: name 'ssl_history' is not defined

## 6. Phase 2 — Fine-Tuning Supervisé

On charge l'encodeur pré-entraîné et on ajoute une tête de classification.

**Stratégie en deux temps :**
1. **Linear probe** (20 epochs) : backbone gelé, on entraîne seulement la tête linéaire → warm-up rapide
2. **Full fine-tune** (130 epochs) : tout dégelé, LR backbone = LR/10

In [6]:
FT_IMG_SIZE  = (384, 576)   # ratio 3:2 préservé
FT_BATCH     = 32
FT_WORKERS   = 0

ft_train_tfms = transforms.Compose([
    transforms.Resize((432, 648)),
    transforms.RandomCrop(FT_IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.10, scale=(0.01, 0.04)),
])

ft_eval_tfms = transforms.Compose([
    transforms.Resize(FT_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


class TildaDataset(Dataset):
    def __init__(self, dataframe=None, image_dir=None, ids=None, transform=None, has_labels=True):
        self.dataframe  = dataframe.reset_index(drop=True) if dataframe is not None else None
        self.image_dir  = Path(image_dir) if image_dir is not None else None
        self.ids        = list(ids) if ids is not None else None
        self.transform  = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.dataframe) if self.has_labels else len(self.ids)

    def __getitem__(self, idx):
        if self.has_labels:
            row      = self.dataframe.iloc[idx]
            image_id = int(row['id'])
            path     = Path(row['path'])
            label    = int(row['label'])
        else:
            image_id = int(self.ids[idx])
            path     = self.image_dir / f'{image_id}.tif'
            label    = -1
        img = Image.open(path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label, image_id


test_ids    = sorted([int(p.stem) for p in test_dir.glob('*.tif')])
test_ds     = TildaDataset(image_dir=test_dir, ids=test_ids, transform=ft_eval_tfms, has_labels=False)
test_loader = DataLoader(test_ds, batch_size=FT_BATCH, shuffle=False,
                         num_workers=FT_WORKERS, pin_memory=True)
print('Test loader:', len(test_loader), 'batches')

Test loader: 25 batches


In [7]:
class SSLFineTuneModel(nn.Module):
    """Encodeur SSL + tête de classification."""
    def __init__(self, num_classes=8, dropout=0.30):
        super().__init__()
        self.encoder = SEResNetEncoder(in_channels=1)
        self.head    = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def load_ssl_weights(self, ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu')
        self.encoder.load_state_dict(ckpt['encoder_state_dict'])
        print(f'Chargé SSL checkpoint (epoch {ckpt["epoch"]}, loss {ckpt["loss"]:.4f})')

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.head(self.encoder(x))

In [8]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for images, labels, _ in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            loss   = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(1).detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


def fine_tune(model, tr_loader, va_loader, fold_name,
              probe_epochs=20, ft_epochs=130,
              probe_lr=0.05, ft_lr=0.005):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_acc  = -1.0
    best_epoch = 0
    history   = []
    best_path = CHECKPOINT_DIR / f'best_{fold_name}.pt'
    start     = time.time()

    # ── Phase 1 : Linear Probe ────────────────────────────────────────
    model.freeze_encoder()
    optimizer = torch.optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=probe_lr, momentum=0.9, weight_decay=1e-4, nesterov=True
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=probe_epochs, eta_min=1e-4)
    print(f'  [{fold_name}] Phase 1: linear probe ({probe_epochs} epochs)')

    for epoch in range(1, probe_epochs + 1):
        tr_loss, tr_acc = run_epoch(model, tr_loader, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, va_loader, criterion)
        scheduler.step()
        if va_acc > best_acc:
            best_acc = va_acc; best_epoch = epoch
            torch.save({'model_state_dict': model.state_dict(), 'best_acc': best_acc}, best_path)
        history.append({'phase': 'probe', 'epoch': epoch,
                        'train_acc': tr_acc, 'val_acc': va_acc,
                        'train_loss': tr_loss, 'val_loss': va_loss})
        print(f'  [probe] ep {epoch:02d} | train {tr_acc:.4f} | val {va_acc:.4f} | best {best_acc:.4f}')

    # ── Phase 2 : Full Fine-Tune ──────────────────────────────────────
    model.unfreeze_encoder()
    optimizer = torch.optim.SGD([
        {'params': model.encoder.parameters(), 'lr': ft_lr * 0.1},
        {'params': model.head.parameters(),    'lr': ft_lr},
    ], momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ft_epochs, eta_min=1e-5)
    patience  = 50
    print(f'  [{fold_name}] Phase 2: full fine-tune ({ft_epochs} epochs)')

    for epoch in range(1, ft_epochs + 1):
        tr_loss, tr_acc = run_epoch(model, tr_loader, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, va_loader, criterion)
        scheduler.step()
        if va_acc > best_acc:
            best_acc = va_acc; best_epoch = epoch + probe_epochs
            torch.save({'model_state_dict': model.state_dict(), 'best_acc': best_acc}, best_path)
        history.append({'phase': 'finetune', 'epoch': probe_epochs + epoch,
                        'train_acc': tr_acc, 'val_acc': va_acc,
                        'train_loss': tr_loss, 'val_loss': va_loss})
        print(f'  [ft]    ep {epoch:03d} | train {tr_acc:.4f} | val {va_acc:.4f} | best {best_acc:.4f} @ {best_epoch}')
        if epoch - (best_epoch - probe_epochs) >= patience:
            print(f'  Early stop — {patience} epochs sans amélioration')
            break

    elapsed = (time.time() - start) / 60
    print(f'  {fold_name}: {elapsed:.1f} min — best val acc {best_acc:.4f}')
    return pd.DataFrame(history), best_path, best_acc, elapsed

## 7. 5-Fold Fine-Tuning

In [9]:
N_SPLITS     = 5
PROBE_EPOCHS = 20
FT_EPOCHS    = 130
PROBE_LR     = 0.05
FT_LR        = 0.005
SSL_CKPT     = CHECKPOINT_DIR / 'ssl_encoder.pt'

# ← change ici pour reprendre depuis un fold spécifique
# Les proba des folds déjà faits sont chargées depuis les .npy sauvegardés
START_FOLD   = 1

skf            = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_results   = []
all_histories  = {}
fold_probs     = []
ids_reference  = None


def make_loaders(tr_df, va_df):
    tr = TildaDataset(tr_df, transform=ft_train_tfms, has_labels=True)
    va = TildaDataset(va_df, transform=ft_eval_tfms,  has_labels=True)
    return (
        DataLoader(tr, batch_size=FT_BATCH, shuffle=True,  num_workers=FT_WORKERS, pin_memory=True),
        DataLoader(va, batch_size=FT_BATCH, shuffle=False, num_workers=FT_WORKERS, pin_memory=True),
    )


def predict_proba(model, loader, use_tta=True):
    model.eval()
    all_probs, all_ids = [], []
    with torch.no_grad():
        for images, _, image_ids in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            logits_list = [model(images)]
            if use_tta:
                logits_list.append(model(torch.flip(images, dims=[-1])))
                logits_list.append(model(torch.flip(images, dims=[-2])))
                logits_list.append(model(torch.flip(images, dims=[-2, -1])))
            probs = torch.stack([torch.softmax(l, dim=1) for l in logits_list]).mean(0)
            all_probs.append(probs.cpu())
            all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs, dim=0).numpy(), np.array(all_ids)


print('=' * 70)
print(f'5-FOLD FINE-TUNING — SimCLR SE-ResNet34 (depuis fold {START_FOLD})')
print('=' * 70)

for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label']), start=1):
    fold_name  = f'simclr_seres34_fold{fold}'
    probs_path = CHECKPOINT_DIR / f'probs_{fold_name}.npy'
    ids_path   = CHECKPOINT_DIR / f'ids_{fold_name}.npy'

    if fold < START_FOLD:
        # Charger les proba sauvegardées pour inclure ce fold dans l'ensemble final
        if probs_path.exists() and ids_path.exists():
            probs = np.load(probs_path)
            ids   = np.load(ids_path)
            fold_probs.append(probs)
            if ids_reference is None:
                ids_reference = ids
            print(f'  Fold {fold} — chargé depuis .npy ({probs.shape})')
        else:
            print(f'  Fold {fold} — SKIP (pas de .npy, relancer avec START_FOLD=1)')
        continue

    print(f'\n── Fold {fold}/5 ──')

    tr_ld, va_ld = make_loaders(
        df.iloc[tr_idx].reset_index(drop=True),
        df.iloc[va_idx].reset_index(drop=True),
    )

    model = SSLFineTuneModel(num_classes=8).to(DEVICE)
    model.load_ssl_weights(SSL_CKPT)

    history, best_path, best_acc, elapsed = fine_tune(
        model, tr_ld, va_ld, fold_name,
        probe_epochs=PROBE_EPOCHS, ft_epochs=FT_EPOCHS,
        probe_lr=PROBE_LR, ft_lr=FT_LR,
    )

    history.to_csv(OUTPUT_DIR / f'history_{fold_name}.csv', index=False)
    all_histories[fold_name] = history

    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    probs, ids = predict_proba(model, test_loader, use_tta=True)

    # Sauvegarde des soft probs pour pouvoir reprendre l'ensemble si interruption
    np.save(probs_path, probs)
    np.save(ids_path, ids)

    if ids_reference is None:
        ids_reference = ids
    else:
        assert np.array_equal(ids_reference, ids)

    fold_probs.append(probs)
    sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1)}).sort_values('id')
    sub.to_csv(SUBMISSION_DIR / f'submission_{fold_name}_tta_labels0.csv', index=False)

    fold_results.append({'fold': fold, 'fold_name': fold_name,
                          'best_val_acc': best_acc, 'training_time_min': elapsed})
    del model
    torch.cuda.empty_cache()

fold_df = pd.DataFrame(fold_results)
fold_df.to_csv(OUTPUT_DIR / 'results_simclr.csv', index=False)
display(fold_df)
print(f'\nMean val acc : {fold_df["best_val_acc"].mean():.4f} ± {fold_df["best_val_acc"].std():.4f}')


5-FOLD FINE-TUNING — SimCLR SE-ResNet34 (depuis fold 1)

── Fold 1/5 ──
Chargé SSL checkpoint (epoch 298, loss 1.1767)
  [simclr_seres34_fold1] Phase 1: linear probe (20 epochs)


  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  [probe] ep 01 | train 0.2262 | val 0.3192 | best 0.3192


  0%|          | 0/59 [00:00<?, ?it/s]

...

  0%|          | 0/15 [00:00<?, ?it/s]

  [ft]    ep 023 | train 0.5368 | val 0.5742 | best 0.6017 @ 42


  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Soumission Kaggle

In [ ]:
ensemble_probs = np.mean(fold_probs, axis=0)
sub = pd.DataFrame({'id': ids_reference, 'label': ensemble_probs.argmax(1)}).sort_values('id')
sub_path = SUBMISSION_DIR / 'submission_simclr_seres34_5fold_tta_labels0.csv'
sub.to_csv(sub_path, index=False)
print('Saved:', sub_path.name)
print(sub['label'].value_counts().sort_index())
display(sub.head(10))

# Courbes de validation
fig, ax = plt.subplots(figsize=(12, 5))
for fold_name, hist in all_histories.items():
    ax.plot(hist['epoch'], hist['val_acc'], label=fold_name, alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val accuracy')
ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'simclr_finetune_val_acc.png', dpi=150)
plt.show()